In [6]:
import csv
import gzip
import re

#generating csv output file with high and moderate impact INDELs from original VEP run

#INDELS input/output files

vep_file = "M4_family_singletonINDELS-VeP.txt"
vcf_file = "M4_family_singletonINDELS.vcf.gz"
output_file = "FINAL_M4_candidate_INDELS_summary.csv"

parentals = {
    "MN1806CN-1_founder-prep04_Rep1-MN1806CN-1_founder-prep04_Rep1-CTTAAGCTTAAG",
    "MN1806CN-1_founder-prep04_Rep2-MN1806CN-1_founder-prep04_Rep2-TGGCTTTGGCTT",
    "MN1806CN-1_founder-prep11_Rep1-MN1806CN-1_founder-prep11_Rep1-GAGTGCGAGTGC",
    "MN1806CN-1_founder-prep11_Rep2-MN1806CN-1_founder-prep11_Rep2-ATGGTGATGGTG",
}


def normalize_indel(chrom, pos, ref, alt):
    """Convert a VCF indel to an unanchored allele representation."""
    pos = int(pos)

    #remove shared prefix and advance position
    while ref and alt and ref[0] == alt[0]:
        ref = ref[1:]
        alt = alt[1:]
        pos += 1

    #remove shared suffix
    while ref and alt and ref[-1] == alt[-1]:
        ref = ref[:-1]
        alt = alt[:-1]

    #VEP represents an empty allele with "-"
    ref = ref if ref else "-"
    alt = alt if alt else "-"

    return (chrom, str(pos), ref, alt)



def variant_key(chrom, pos, ref, alt):
    """Exact VCF-style variant key."""
    return (chrom, str(pos), ref, alt)


def parse_uploaded_variation(uploaded):
    """
    Parse common VEP Uploaded_variation IDs such as:
        Gm01_12345_A/AT
        1_12345_AT/A

    Returns (chrom, pos, ref, alt), or None if not parseable.
    """
    match = re.match(r"^(.+)_([0-9]+)_([^/]+)/([^/]+)$", uploaded)

    if not match:
        return None

    chrom, pos, ref, alt = match.groups()
    return variant_key(chrom, pos, ref, alt)

#use same depth and GQ filtering parameters as before
MIN_DP = 10
MAX_DP = 100
MIN_GQ = 20

def classify_ad(ad, gq):
    """Classify allele balance using DP and GQ filters"""

    if ad in (None, "", ".", "./.") or gq in (None, "", "."):
        return None, None, None, None

    parts = ad.split(",")

    if len(parts) != 2:
        return None, None, None, None

    try:
        ref_reads = int(parts[0])
        alt_reads = int(parts[1])
        gq_value = float(gq)
    except (ValueError, TypeError):
        return None, None, None, None

    dp = ref_reads + alt_reads

    if dp < 10 or dp > 100 or gq_value < 20:
        return None, ref_reads, alt_reads, None

    if dp == 0:
        return None, ref_reads, alt_reads, None

    ab = alt_reads / dp

    if ab < 0.4:
        call = "REF"
    elif ab < 0.9:
        call = "HET"
    else:
        call = "HOM_ALT"

    return ab, ref_reads, alt_reads, call


#Read vep annotations for the INDELS

variants = {}
vep_fallback_keys = {}

with open(vep_file, "r") as f:
    for line in f:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) < 14:
            continue

        uploaded = fields[0]
        location = fields[1]
        allele = fields[2]
        gene = fields[3]
        consequence = fields[6]
        protein_position = fields[9]
        amino_acids = fields[10]
        codons = fields[11]
        extra = fields[13]

        if "IMPACT=HIGH" in extra:
            impact = "HIGH"
        elif "IMPACT=MODERATE" in extra:
            impact = "MODERATE"
        else:
            continue

        symbol_match = re.search(r"(?:^|;)SYMBOL=([^;]+)", extra)
        symbol = symbol_match.group(1) if symbol_match else ""

        annotation = {
            "gene": gene,
            "symbol": symbol,
            "consequence": consequence,
            "impact": impact,
            "protein_position": protein_position,
            "amino_acids": amino_acids,
            "codons": codons,
            "vep_allele": allele,
        }

        #reconstruct exact original VCF allele
        parsed_key = parse_uploaded_variation(uploaded)

        if parsed_key is not None:
            chrom_v, pos_v, ref_v, alt_v = parsed_key

        normalized_key = normalize_indel(chrom_v, pos_v, ref_v, alt_v)

        variants[normalized_key] = annotation

print("VEP annotations loaded:", len(variants))
print("VEP fallback annotations loaded:", len(vep_fallback_keys))


#Read the vcf and summarize the non-parental carriers 

with gzip.open(vcf_file, "rt") as vcf, \
     open(output_file, "w", newline="") as out:

    writer = csv.writer(out)

    writer.writerow([
        "Gene_ID",
        "Gene_symbol",
        "Chromosome",
        "Position",
        "REF",
        "ALT",
        "Indel_type",
        "Indel_size_bp",
        "Consequence",
        "Impact",
        "Protein_position",
        "AA_change",
        "Codon_change",
        "Num_carriers",
        "HET_samples",
        "HOM_ALT_samples",
        "Carrier_details",
    ])

    samples = []
    parental_indices = []
    mutant_indices = []

    n_biallelic_indels = 0
    n_matching_vep_sites = 0
    n_skipped_parental_sites = 0
    n_written_sites = 0
    n_unmatched_vep = 0

    for line in vcf:
        if line.startswith("##"):
            continue

        if line.startswith("#CHROM"):
            header = line.rstrip("\n").split("\t")
            samples = header[9:]
            sample_index = {sample: i for i, sample in enumerate(samples)}

            found_parentals = parentals.intersection(samples)
            missing_parentals = parentals - set(samples)

            print("Samples in VCF:", len(samples))
            print("Parental samples found:", len(found_parentals))

            if missing_parentals:
                raise ValueError(
                    "These parental sample names were not found:\n"
                    + "\n".join(sorted(missing_parentals))
                )

            parental_indices = [
                sample_index[s] for s in parentals
            ]
            mutant_indices = [
                i for i, sample in enumerate(samples)
                if sample not in parentals
            ]
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) < 10:
            continue

        chrom, pos, ref, alt = fields[0], fields[1], fields[3], fields[4]

        #exclude multiallelic sites
        if "," in alt:
            continue

        #keep indels only. skip SNPs and equal-length substitutions if any are present
        indel_size = abs(len(alt) - len(ref))

        if indel_size == 0:
            continue

        n_biallelic_indels += 1

        indel_type = (
            "Insertion" if len(alt) > len(ref)
            else "Deletion"
        )

        key = normalize_indel(chrom, pos, ref, alt)
        annotation = variants.get(key)

        #cases where VEP's allele matches the VCF ALT
        if annotation is None:
            annotation = vep_fallback_keys.get((chrom, pos, alt))

        if annotation is None:
            n_unmatched_vep += 1
            continue

        n_matching_vep_sites += 1

        format_fields = fields[8].split(":")

        if "AD" not in format_fields:
            continue

        ad_index = format_fields.index("AD")
        gt_index = (
            format_fields.index("GT")
            if "GT" in format_fields else None
        )
        gq_index = format_fields.index("GQ") if "GQ" in format_fields else None

        calls = {}

        for i, sample in enumerate(samples):
            sample_values = fields[9 + i].split(":")

            ad = (
                sample_values[ad_index]
                if ad_index < len(sample_values) else "."
            )
            gt = (
                sample_values[gt_index]
                if gt_index is not None and gt_index < len(sample_values)
                else "."
            )
            gq = (
                sample_values[gq_index]
                if gq_index is not None and gq_index < len(sample_values)
                else "."
            )

            ab, ref_reads, alt_reads, call = classify_ad(ad, gq)

            calls[sample] = {
                "ab": ab,
                "ref_reads": ref_reads,
                "alt_reads": alt_reads,
                "call": call,
                "gt": gt,
            }


        #exclude the entire site if any parent has confident ALT evidence
        parental_alt_samples = [
            samples[i]
            for i in parental_indices
            if calls[samples[i]]["call"] in ("HET", "HOM_ALT")
        ]

        if parental_alt_samples:
            n_skipped_parental_sites += 1
            continue

        het_samples = []
        hom_samples = []
        carrier_details = []
        family_carriers = {}

        for i in mutant_indices:
            sample = samples[i]
            info = calls[sample]

            if info["call"] not in ("HET", "HOM_ALT"):
                continue

            match = re.match(r"^(MEE\d+)", sample)
            if match is None:
                raise ValueError(
                    f"Cannot determine family ID from sample name: {sample}"
                )

            family_id = match.group(1)
            family_carriers.setdefault(family_id, []).append(
                (sample, info)
            )

        #keep sites carried by exactly one mutant family
        if len(family_carriers) != 1:
            continue

        for family_samples in family_carriers.values():
            for sample, info in family_samples:
                if info["call"] == "HET":
                    het_samples.append(sample)
                else:
                    hom_samples.append(sample)

                carrier_details.append(
                    f"{sample}(GT={info['gt']},"
                    f"AD={info['ref_reads']},{info['alt_reads']},"
                    f"AB={info['ab']:.3f},{info['call']})"
                )

        carriers = het_samples + hom_samples

        if not carriers:
            continue

        writer.writerow([
            annotation["gene"],
            annotation["symbol"],
            chrom,
            pos,
            ref,
            alt,
            indel_type,
            indel_size,
            annotation["consequence"],
            annotation["impact"],
            annotation["protein_position"],
            annotation["amino_acids"],
            annotation["codons"],
            len(carriers),
            ";".join(het_samples),
            ";".join(hom_samples),
            "; ".join(carrier_details),
        ])

        n_written_sites += 1

print()
print("Finished.")
print("Biallelic indels examined:", n_biallelic_indels)
print("Indels matched to VEP annotations:", n_matching_vep_sites)
print("Indels unmatched to VEP annotations:", n_unmatched_vep)
print("Sites excluded due to parental ALT evidence:", n_skipped_parental_sites)
print("Sites written with at least one mutant carrier:", n_written_sites)
print("Output:", output_file)

VEP annotations loaded: 441
VEP fallback annotations loaded: 0
Samples in VCF: 175
Parental samples found: 4

Finished.
Biallelic indels examined: 13657
Indels matched to VEP annotations: 441
Indels unmatched to VEP annotations: 13216
Sites excluded due to parental ALT evidence: 0
Sites written with at least one mutant carrier: 441
Output: FINAL_M4_candidate_INDELS_summary.csv
